# Classification pipeline

## Prerequisites
Create an environment:

```
conda create -n text_classification python=3.14
conda activate text_classification
```

Install required packages:

```
conda install pandas numpy matplotlib scikit-learn 
conda install -c conda-forge imbalanced-learn sentence_transformers
```



In [1]:
# Importing
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, learning_curve
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score, ConfusionMatrixDisplay
from sklearn.base import BaseEstimator, TransformerMixin
from sentence_transformers import SentenceTransformer


warnings.filterwarnings("ignore")

## Specifying variables
`CHOSEN_MODEL`: set to `None` to compare all models, or set to a model name
(e.g. `"Logistic Regression (TF-IDF)"`) to skip comparison and use that model directly.

`DATA_PATH`: path to your labeled Excel file. <br>
`DATA_PATH_NEW`: path to your unlabeled Excel file for prediction. <br>
`TEXT_COL`: column name containing the utterances. <br>
`LABEL_COL`: column name containing the labels.

`CONFIDENCE_THRESHOLD`: minimum confidence required to auto-classify (0–1). <br>
`CONTEXT_WINDOW`: number of surrounding utterances to include per side (0 = none).

`LABEL_MAP`: maps your short codes to full category names. <br>
`CATEGORIES`: full list of category names in display order. 

`SBERT_MODEL`: HuggingFace model name for sentence embeddings. <br>
`SBERT_CACHE`: filename where embeddings are cached so they are only computed once.
`SBERT_CACHE_UNLABELED`: filename where embeddings are cached so they are only computed once.

In [2]:
CHOSEN_MODEL = None # e.g "Logistic Regression (Tfidf)"
DATA_PATH = "data.xlsx"        
DATA_PATH_NEW = "data_unlabeled.xlsx"
TEXT_COL = "Utterance"               
LABEL_COL = "Label" 

CONFIDENCE_THRESHOLD = 0.7
CONTEXT_WINDOW       = 1

LABEL_MAP = {
    "o":           "Open suggestion",
    "c":           "Closed suggestion",
    "none":        "None",
    "nan":         "None",
    "":            "None",
}

CATEGORIES = ["Open suggestion", "Closed suggestion", "None"]
SBERT_MODEL = "NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers"
SBERT_CACHE           = "sbert_embeddings.npy"
SBERT_CACHE_UNLABELED = "sbert_embeddings_unlabeled.npy" 

In [3]:
def load_data(path):
    """
    Load and combine all sheets from an Excel file into one DataFrame.

    Reads each sheet, adds a Participant column, concatenates all sheets,
    applies label mapping, and removes empty rows.

    Parameters
    ----------
    path: str
        relative path of where the data is located. For example: "data/data.xlsx"

    Returns
    ------------
    pandas.DataFrame
        Cleaned DataFrame with columns: TEXT_COL, LABEL_COL, Participant.
        Labels are mapped to full category names.
        Rows with empty text are removed.

   """
    
    # Combine excel sheets to one combined dataframe  
    xl = pd.ExcelFile(path)
    dfs = []
    for sheet_name in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet_name)
        df["Participant"] = sheet_name # Add column with participant number
        dfs.append(df)
    
    # Make dataframe
    df = pd.concat(dfs, ignore_index=True)
    df.to_csv("data_combined.csv", index=False)
    df.columns = df.columns.str.strip()

    # Assertions
    assert TEXT_COL  in df.columns, f"Column '{TEXT_COL}' not found."
    assert LABEL_COL in df.columns, f"Column '{LABEL_COL}' not found."
    
    # Cleaning data
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip() # Clean text column
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower() # Clean label column
    df[LABEL_COL] = df[LABEL_COL].map(LABEL_MAP).fillna("None") # Fill in every unlabeled Utterance as None, if you don't want this comment this line out.
    df = df[df[TEXT_COL].str.len() > 0].copy() # Remove every empty row that has no text

    # Print statements
    print(df["Participant"].value_counts())
    print("\nClass distribution in percentage per participant:")
    print(df.groupby("Participant")[LABEL_COL].value_counts(normalize=True).unstack(fill_value=0).round(2))
    print("\nTotal rows : ", len(df))
    print("\nClass distribution:\n", df[LABEL_COL].value_counts())
    return df
 

In [4]:
def load_unlabeled_data(path):
    """
    Load unlabeled utterances from an Excel file for prediction.

    Parameters
    ----------
    path: str
        Path to the unlabeled Excelfile. For example: "data/unlabeled_data.xlsx"

    Returns
    ------------
    df: pandas.DataFrame
        A cleaned DataFrame containing all sheets combined.

   """
        
    # Combine sheets in excel to one combined dataframe  
    xl = pd.ExcelFile(path)
    dfs = []
    for sheet_name in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet_name)
        df["Participant"] = sheet_name
        dfs.append(df)

    # Make dataframe
    df = pd.concat(dfs, ignore_index=True)
    df.to_csv("data_combined_unlabeled.csv", index=False)
    df.columns = df.columns.str.strip()

    # Assertions
    assert TEXT_COL  in df.columns, f"Column '{TEXT_COL}' not found."
    
    # Cleaning data
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df = df[df[TEXT_COL].str.len() > 0].copy()

    # Print statements
    print("\nTotal rows : ", len(df))
    return df

In [5]:
def add_context(df, window=CONTEXT_WINDOW):
    """
    Adds context to each utterance by concatenating utterances within a specified window.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the utterances in TEXT_COL.
    window : int, default=CONTEXT_WINDOW
        Number of utterances to include before and after the current utterance.

    Returns
    -------
    with_context: list of str
        List of utterances with added context. Each element contains the current utterance and its surrounding context separated by
        "[SEP]". If window is 0, the original utterances are returned.
   """
    # Return original dataframe if the context window is 0
    if window == 0:
        return df[TEXT_COL].tolist()
        
    # Make text columns into a list
    text = df[TEXT_COL].tolist()
    with_context = []
    
    # For every utterance concatenate with specified amount of utterances around it
    for i in range(len(text)):
        start = max(0, i - window) # To ensure the first utterance won't raise an error
        end   = min(len(text), i + window + 1) # To ensure the last utterance won't raise an error
        with_context.append(" [SEP] ".join(text[start:end])) # Append the right utterances to the with_context list and seperate them with [SEP]
    return with_context
 
 

In [6]:
def compute_sbert_embeddings(texts, model_name=SBERT_MODEL, cache_path=SBERT_CACHE):
    """
    Compute sBERT sentence embeddings, loading from cache when possible.
    Converts text into semantic embeddings using a pre-trained
    Dutch sentence transformer. Unlike TF-IDF, embeddings capture
    meaning rather than word frequency, which is useful when two
    sentences can use different words to express the same idea.

    Embeddings are expensive to compute, this function saves them to disk
    after the first run and reloads them on next runs. If your texts
    change (more data added, context window changed), delete the cache files
    and rerun.

    Parameters
    ----------
    texts : list of str
        Utterances to embed (with context already applied).
    model_name : str
        HuggingFace sentence transformer model to use.
    cache_path : str
        Path where embeddings are saved as a .npy file.

    Returns
    -------
    np.ndarray of shape (n_samples, embedding_dim)
        One embedding vector per utterance.
    """
    texts_cache = cache_path.replace(".npy", "_texts.npy")

    # Load from cache if texts have not changed
    if os.path.exists(cache_path) and os.path.exists(texts_cache):
        cached_texts = np.load(texts_cache, allow_pickle=True).tolist()
        if cached_texts == texts:
            return np.load(cache_path)
        
    # Compute embeddings
    model      = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

    # Save to cache
    np.save(cache_path, embeddings)
    np.save(texts_cache, np.array(texts, dtype=object))
    print(f"Embeddings cached -> {cache_path}")
    return embeddings

In [7]:
def build_tfidf_pipelines():
    """
    Build all classification pipelines to compare.

    Each pipeline consists of a TF-IDF vectorizer followed by a classification algorithm. 
    The TF-IDF vectorizer uses: Unigrams and bigrams and log-scale term frequency.
    Class_weight='balanced' compensates for inbalanced classes.

    Returns
    -------
    dict
        Dictionary mapping model names to scikit-learn Pipeline objects.
    """

    # Parameters used for each Tfidf vectorizer
    tfidf_params = dict(      
        ngram_range=(1, 2),   # Unigrams and bigrams   
        sublinear_tf=True,    # log-scale term frequency
    )
    return {
        # TF-IDF pipelines
        "SVM linear (Tfidf)": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf",   SVC(kernel="linear", probability = True, class_weight="balanced", C=1.0)),
        ]),
        "Logistic Regression (Tfidf)": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf",   LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")),
        ]),
         "Balanced Random Forest (Tfidf)": Pipeline([
            ("tfidf", TfidfVectorizer(**tfidf_params)),
            ("clf", BalancedRandomForestClassifier(n_estimators=300,random_state=42)),
         ]),  
    }

def build_sbert_pipelines():
    """
    Build sBERT classification pipelines.

    These pipelines receive pre-computed embedding vectors directly
    no vectorizer step is needed because embeddings are computed once
    outside the pipeline via compute_sbert_embeddings().

    Returns
    -------
    dict
        Mapping of model name to sklearn Pipeline (classifier only).
    """
    return {
        "SVM linear (sBERT)": Pipeline([
            ("clf", SVC(kernel="linear", probability=True, class_weight="balanced", C=1.0)),
        ]),
        "Logistic Regression (sBERT)": Pipeline([
            ("clf", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")),
        ]),
        "Balanced Random Forest (sBERT)": Pipeline([
            ("clf", BalancedRandomForestClassifier(n_estimators=300, random_state=42)),
        ]),
    }
 
 

In [8]:
def compare_classifiers(pipelines, X, y, n_splits=5):
    """
    Compare multiple classifiers using stratified k-fold cross-validation.
    The classifier with the highest mean macro F1 score is selected as
    the best-performing model.

    Parameters
    ----------
    pipelines : dict
        Dictionary containing model names as keys and scikit-learn
        Pipeline objects as values.

    X : list of str
        Input texts

    y : list of str
        Target labels corresponding to X.

    n_splits : int, default=5
        Number of folds used in stratified cross-validation.

    Returns
    -------
    best_model: str
        Name of the classifier with the highest mean macro F1 score.
    best_score: float
        f1-macro score of the best model

    """
    # Cross-validation
    cv      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = {}
    
    # Print statements for clarity
    print(f"Comparing classifiers ({n_splits}-fold stratified CV)")    
 
    # For every pipeline compute the f1_macro and accuracy of the cross validated results. 
    for name, pipe in pipelines.items():
        scores = cross_validate(pipe, X, y, cv=cv, scoring=["f1_macro", "accuracy"])
        results[name] = {
            "f1_macro": scores["test_f1_macro"].mean(),
            "f1_std":   scores["test_f1_macro"].std(),
            "accuracy": scores["test_accuracy"].mean(),
        }

        # Print the f1 macro: non weighted avergage over all categories and the accuracy
        print(f"\n{name}")
        print(f"  F1 macro : {results[name]['f1_macro']:.3f} ± {results[name]['f1_std']:.3f}")
        print(f"  Accuracy : {results[name]['accuracy']:.3f}")

    best_model = None
    best_score = -1

    # Find best model based on f1_macro and save model_name as best_model
    for model_name, metrics in results.items():
        if metrics["f1_macro"] > best_score:
            best_score = metrics["f1_macro"]
            best_model = model_name

    # print best model with its results
    print("\n -> Best model:", best_model, f"(F1 macro = {results[best_model]['f1_macro']:.3f})\n")

    return best_model, best_score

In [9]:
def detailed_report(pipeline, X, y):
    """
    Generate a detailed evaluation report for one pipeline including: Precision, recall, and F1-score per class, 
    Cohen's Kappa agreement score and Confusion matrix visualization

    Parameters
    ----------
    pipeline : sklearn.pipeline.Pipeline
        Classification pipeline to evaluate.

    X : list of str
        Input texts

    y : list of str
        True labels

    Returns
    -------
    None
        Prints the report and saves confusion_matrix.png.
    """

    # Predict y with cross validation
    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)

    # Print classification report with precision, recall, f1-score, support for each category and accuracy, macro average and weighted average
    print(classification_report(y, y_pred, labels=CATEGORIES, zero_division=0))

    # Compute cohens kappa between the given labels and the predicted labels
    kappa = cohen_kappa_score(y, y_pred)
    print(f"Cohen's Kappa (human vs model): {kappa:.3f}")

    # Compute confusion matrix
    cm = confusion_matrix(y, y_pred, labels=CATEGORIES, normalize = 'true')

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=CATEGORIES).plot(ax=ax, colorbar=False, xticks_rotation=20)
    ax.set_title("Confusion Matrix (cross-validated)")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.show()
    print("Confusion matrix saved -> confusion_matrix.png\n")
 

In [10]:
def plot_learning_curve(pipeline, X, y, train_sizes=np.linspace(0.10, 0.99, 20), n_splits=5, thresholds=None):
    """
    Plot validation F1 macro against training set size for one pipeline.

    Shows how model performance grows as more labeled data is added. 
    Shaded area shows ± 1 standard deviation across folds.

    Parameters
    ----------
    pipeline : sklearn Pipeline
        The pipeline to evaluate. Should not be fitted yet.
    X : np.array
        Input texts.
    y : np.array
        Target labels.
    train_sizes : array-like
        Fractions of training data to test, e.g. np.linspace(0.10, 0.99, 20).
    n_splits : int, default=5
        Number of cross-validation folds at each training size.
    thresholds : list of float or None
        Horizontal lines drawn at these F1 values (e.g. [0.60, 0.70]).
        If None, no threshold lines are drawn.

    Returns
    -------
    None
        Displays and saves learning_curve.png.
    """
    if thresholds is None:
        thresholds = []
        
    # Cross validation
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Generate learning curves
    sizes_abs, train_scores, val_scores = learning_curve(pipeline, X, y, train_sizes=train_sizes, cv=cv, scoring="f1_macro", n_jobs=1)
    
    # Mean and standard deviation over the k-folds of the training and the validation
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    # Plotting
    fig, ax = plt.subplots(figsize=(9, 5))

    # Plot training 
    ax.plot(sizes_abs, train_mean, label="Training F1",
            color="blue", linewidth=2, marker="o", markersize=4)
    ax.fill_between(sizes_abs, train_mean - train_std, train_mean + train_std,
                    alpha=0.10, color="aqua")
    # Plot validation
    ax.plot(sizes_abs, val_mean, label="Validation F1",
            color="green", linewidth=2, marker="o", markersize=4)
    ax.fill_between(sizes_abs, val_mean - val_std, val_mean + val_std,
                    alpha=0.10, color="lawngreen")

    # Formatting
    for t in thresholds:
        ax.axhline(y=t, linestyle="--", color="grey", linewidth=1.2, label=f"F1 = {t:.2f}")
        
    ax.set_xlabel("Number of training examples", fontsize=12)
    ax.set_ylabel("F1 macro", fontsize=12)
    ax.set_title("Learning Curve", fontsize=14)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("learning_curve.png", dpi=150)
    plt.show()
    print("Learning curve saved -> learning_curve.png")

    # Threshold crossings
    if thresholds:
        print("\nMinimum training examples to reach each threshold:")
        for t in thresholds:
            crossed = np.where(val_mean >= t)[0]
            if len(crossed) > 0:
                print(f"  F1 ≥ {t:.2f} : {int(sizes_abs[crossed[0]])}")
            else:
                print(f"  F1 ≥ {t:.2f} : never")

In [11]:
def predict_new(pipeline, df_new, X_new_sbert=None):
    """
    Classifies unlabeled utterances and flags uncertain predictions for human review

    Parameters 
     ----------

    pipeline : fitted sklearn Pipeline
        Classification pipeline used to predict new labels
    df_new : pandas.DataFrame
        Rows to classify, must contain TEXT_COL
    X_new_sbert : np.ndarray or None
        Pre-computed sBERT embeddings for df_new. Pass this when using
        an sBERT pipeline so embeddings are not recomputed. Leave as None
        for TF-IDF pipelines.
    
     Returns
     ----------
     df_out: pd.DataFrame
         
    
    """
    if df_new.empty:
        print("No unlabeled sentences to predict.")
        return pd.DataFrame()
        
    if TEXT_COL not in df_new.columns:
        raise ValueError(f"Column '{TEXT_COL}' not found in input DataFrame.")
    
    # Predict probabilities 
    # Predict input
    if X_new_sbert is not None:
        X_new = X_new_sbert
    else:
        X_new = add_context(df_new)
        
    proba  = pipeline.predict_proba(X_new)
    
    # Get best class plus confidance
    pred_idx = np.argmax(proba, axis=1)
    labels = pipeline.classes_[np.argmax(proba, axis=1)]
    conf = proba[np.arange(len(proba)), pred_idx]
    
    # Output dataframe 
    df_out = df_new[["Participant", TEXT_COL]].copy().reset_index(drop=True)
    df_out["predicted_label"]    = labels
    df_out["confidence"]         = conf.round(3)
    df_out["needs_human_review"] = conf < CONFIDENCE_THRESHOLD

    # Add per category propobility columns
    for i, cat in enumerate(pipeline.classes_):
        df_out[f"p_{cat}"] = proba[:, i].round(3)

    # Summary
    n      = len(df_out)
    auto   = (conf >= CONFIDENCE_THRESHOLD).sum()
    review = (conf <  CONFIDENCE_THRESHOLD).sum()
    

    print(f"Sentences predicted : {n}")
    print(f"  Auto-classified    : {auto}  ({100*auto/n:.0f}%)\n")
    print(f"  Flagged for review : {review}  ({100*review/n:.0f}%)\n")
    print(f"  Confidence threshold : {CONFIDENCE_THRESHOLD}")
    print(df_out["Participant"].value_counts())
    print("\nClass distribution of predicted labels in percentage per participant:")
    print(df_out.groupby("Participant")["predicted_label"].value_counts(normalize=True).unstack(fill_value=0).round(2))
    print("\nClass distribution of predicted labels:\n", df_out["predicted_label"].value_counts())
    return df_out

## Main

In [ ]:
def main():
    
    # 1. Load data
    df = load_data(DATA_PATH)
    df_unlabeled = load_unlabeled_data(DATA_PATH_NEW)
    
    X  = add_context(df)
    y  = df[LABEL_COL].tolist()

    X_unlabeled = add_context(df_unlabeled)

    # Compute sBERT embeddings once (to save memory)
    X_sbert = compute_sbert_embeddings(X)
    
    # 3. Build pipelines
    tfidf_pipelines = build_tfidf_pipelines()
    sbert_pipelines = build_sbert_pipelines()

    if CHOSEN_MODEL is None:
        # 4a. Compare all models    
        print("\nTF-IDF pipelines")
        best_tfidf, best_score_tfidf = compare_classifiers(tfidf_pipelines, X, y)

        print("\nsBERT pipelines")
        best_sbert, best_score_sbert = compare_classifiers(sbert_pipelines, X_sbert, y)

        # 5a. Compare tfidf with sBERT
        if best_score_tfidf >= best_score_sbert:
            best_name     = best_tfidf
            best_pipeline = tfidf_pipelines[best_name]
            X_best        = X
            print(f"\n-> Overall best: {best_name} (F1={best_score_tfidf:.3f})")
        else:
            best_name     = best_sbert
            best_pipeline = sbert_pipelines[best_name]
            X_best        = X_sbert
            print(f"\n -> Overall best: {best_name} (F1={best_score_sbert:.3f})")
 

        # 6a. Detailed report for best model
        print(f"Detailed report of {best_name}")
        detailed_report(best_pipeline, X_best, y)
    
        # 7a. Learning curve for best model
        plot_learning_curve(
            best_pipeline, X_best, y,
            train_sizes=np.linspace(0.01, 1, 25),
            thresholds=[0.50, 0.55, 0.60, 0.65, 0.70],
        )   
    else:
        # 4b. Use chosen model
        all_pipelines = {**tfidf_pipelines, **sbert_pipelines}

        best_pipeline = all_pipelines[CHOSEN_MODEL]
        print(f"Using chosen model: {CHOSEN_MODEL}")

    
    # 7. Train on all labeled data
    if "sBERT" in (CHOSEN_MODEL or best_name):
        X_best = X_sbert
        X_unlabeled_sbert = X_unlabeled_sbert = compute_sbert_embeddings(X_unlabeled, cache_path=SBERT_CACHE_UNLABELED)
    else:
        X_best = X
        X_unlabeled_sbert = None

    print(f"\nUsing chosen model: {CHOSEN_MODEL}")
    best_pipeline.fit(X_best, y)
 
    # 6. Predict new sentences
    predictions = predict_new(best_pipeline, df_unlabeled, X_unlabeled_sbert)
 
    if not predictions.empty:
        predictions.to_csv("predictions.csv", index=False)
        print("Predictions saved -> predictions.csv")
    
if __name__ == "__main__":
    main()

Participant
11    399
20    335
10    295
5     248
2     221
32    213
7     209
12    197
6      97
Name: count, dtype: int64

Class distribution in percentage per participant:
Label        Closed suggestion  None  Open suggestion
Participant                                          
10                        0.05  0.81             0.14
11                        0.09  0.80             0.11
12                        0.07  0.78             0.15
2                         0.19  0.75             0.07
20                        0.14  0.77             0.09
32                        0.15  0.64             0.22
5                         0.04  0.88             0.08
6                         0.10  0.79             0.10
7                         0.06  0.86             0.08

Total rows :  2214

Class distribution:
 Label
None                 1747
Open suggestion       251
Closed suggestion     216
Name: count, dtype: int64

Total rows :  533


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Embeddings cached -> sbert_embeddings.npy

TF-IDF pipelines
Comparing classifiers (5-fold stratified CV)
